# Project 04: Chatbot - NLP Intent Classification 🤖

## 🎯 Learning Objectives

- Master text preprocessing (tokenization, stemming)
- Apply Bag of Words feature extraction
- Train classifiers for intent recognition
- Build a conversational chatbot interface

**Difficulty:** Intermediate | **Time:** 3-4 hours

---

## Phase 1: Setup & Import Dependencies

In [15]:
# ✅ WORKING EXAMPLE: Import basic libraries
import numpy as np
import pandas as pd
import json
import random
from datetime import datetime

print("✅ Basic libraries imported!")

✅ Basic libraries imported!


In [16]:
# ✅ WORKING EXAMPLE: Dataset is included in the template!
# No download needed - the intents.json file is in ./data/ folder
print("📊 Dataset location: ./data/intents.json")
print("✅ Dataset ready to use!")

📊 Dataset location: ./data/intents.json
✅ Dataset ready to use!


In [17]:
# ✅ WORKING EXAMPLE: Import NLTK and download required data
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
from nltk.stem import snowball

# Declare the Snowball Stemmer for English
snowballStemmer = snowball.SnowballStemmer("english")
print("✅ NLTK ready!")

✅ NLTK ready!


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


---

## Phase 2: Text Preprocessing Function 🔧

Text preprocessing converts raw text into a clean format:
1. **Tokenize**: Break sentence into words
2. **Stem**: Reduce words to root form ("running" → "run")
3. **Clean**: Remove punctuation and special characters

### ⚠️ TODO 1: Complete Text Preprocessing Function (Medium)

**Your Task:** Complete the text_preprocessing function.

**Steps:**
1. Tokenize the sentence using `nltk.word_tokenize(sentence)`
2. Loop through tokens and stem each one using `snowballStemmer.stem(token.lower())`
3. Only keep tokens that are alphanumeric (use `.isalnum()`)
4. Join the stemmed tokens into a string and return it

**Expected Output for test:**
```
Input: 'We all agreed, it was a magnificent evening.'
Output: 'we all agre it was a magnific even'
```

In [18]:
# ⚠️ TODO 1: Complete the text_preprocessing function
#
def text_preprocessing(sentence):
    # Step 1: Tokenize the sentence
    tokens = nltk.word_tokenize(sentence)

    # Step 2: Stem each token and filter
    stem_tokens = []
    for token in tokens:
        token = token.lower()
        if token.isalnum():
            stem_tokens.append(snowballStemmer.stem(token))

    # Step 3: Join and return
    return " ".join(stem_tokens)

# Test the function
test_sentence = 'We all agreed, it was a magnificent evening.'
result = text_preprocessing(test_sentence)
print(f"Input: '{test_sentence}'")
print(f"Output: '{result}'")

Input: 'We all agreed, it was a magnificent evening.'
Output: 'we all agre it was a magnific even'


---

## Phase 3: Load and Process Dataset 📊

In [19]:
# ✅ WORKING EXAMPLE: Load JSON data from local file or GitHub
# =============================================================================
# 📊 Data Loading - Works Locally, Browser Colab, AND VS Code + Colab!
# =============================================================================
import os
import urllib.request

DATA_FILE = './data/intents.json'
DATA_URL = 'https://raw.githubusercontent.com/KCW89/telebort-public-datasets/main/AI-2/csv/intents.json'

if os.path.exists(DATA_FILE):
    print("📊 Loading dataset from local file...")
    with open(DATA_FILE) as f:
        data = json.load(f)
else:
    print("📥 Local file not found. Downloading from GitHub...")
    os.makedirs("./data", exist_ok=True)
    
    urllib.request.urlretrieve(DATA_URL, DATA_FILE)
    print("✅ Downloaded and cached locally!")
    
    with open(DATA_FILE) as f:
        data = json.load(f)

# Declare data structures
intent_list = []      # All unique intents
train_data = []       # Preprocessed training texts
train_label = []      # Intent labels for each text
responses = {}        # Intent -> list of responses mapping

print(f"✅ Loaded {len(data['intents'])} intents from dataset")
print(f"Sample intent: {data['intents'][0]['intent']}")

📊 Loading dataset from local file...
✅ Loaded 13 intents from dataset
Sample intent: Greeting


### ⚠️ TODO 2: Process Dataset (Medium)

**Your Task:** Loop through the dataset and populate train_data and train_label.

**Steps:**
1. For each intent in data['intents']:
   - Add intent name to intent_list
   - Store responses in responses dictionary
   - For each text in intent['text']:
     - Preprocess the text
     - Append to train_data
     - Append intent name to train_label

**Expected Output:**
```
train_data[2:5]: ['hola', 'hello', 'hello there']
train_label[2:5]: ['Greeting', 'Greeting', 'Greeting']
```

In [20]:
# ⚠️ TODO 2: Process the dataset
#
# for intent in data['intents']:
#     # Add intent to list
#     intent_list.append(intent['intent'])
#     # Store responses
#     responses[intent['intent']] = intent['responses']
#
#     # Process each text example
#     for text in intent['text']:
#         preprocessed_text = text_preprocessing(???)
#         train_data.append(???)
#         train_label.append(intent['???'])
#
# print(f"Intents: {intent_list}")
# print(f"\ntrain_data[2:5]: {train_data[2:5]}")
# print(f"train_label[2:5]: {train_label[2:5]}")
# print(f"\nResponses for 'Thanks': {responses['Thanks']}")
#
# YOUR CODE HERE:
for intent in data['intents']:
    # Add intent to list
    intent_list.append(intent['intent'])
    # Store responses
    responses[intent['intent']] = intent['responses']

    # Process each text example
    for text in intent['text']:
        preprocessed_text = text_preprocessing(text)
        train_data.append(preprocessed_text)
        train_label.append(intent['intent'])

print(f"Intents: {intent_list}")
print(f"\ntrain_data[2:5]: {train_data[2:5]}")
print(f"train_label[2:5]: {train_label[2:5]}")
print(f"\nResponses for 'Thanks': {responses['Thanks']}")

Intents: ['Greeting', 'Goodbye', 'Thanks', 'Apology', 'Name', 'Age', 'TimeQuery', 'Jokes', 'Help', 'Weather', 'Compliment', 'Feeling', 'noanswer']

train_data[2:5]: ['hola', 'hello', 'hello there']
train_label[2:5]: ['Greeting', 'Greeting', 'Greeting']

Responses for 'Thanks': ["You're welcome!", 'Happy to help!', 'Anytime!', 'My pleasure!', 'Glad I could help!']


---

## Phase 4: Feature Extraction (Bag of Words) 🎒

Convert text to numbers using CountVectorizer!

In [21]:
# ✅ WORKING EXAMPLE: Import CountVectorizer
from sklearn.feature_extraction.text import CountVectorizer

# Declare the vectorizer
vectorizer = CountVectorizer()
print("✅ CountVectorizer ready!")

✅ CountVectorizer ready!


### ⚠️ TODO 3: Create Vocabulary and Bag of Words (Easy)

**Your Task:** Fit the vectorizer and transform train_data.

**Steps:**
1. Fit vectorizer on train_data: `vectorizer.fit(train_data)`
2. Get vocabulary: `list_of_words = vectorizer.get_feature_names_out()`
3. Transform to BOW: `train_data_bow = vectorizer.transform(train_data)`

**Expected:** Vocabulary list and converted vectors

In [22]:
# ⚠️ TODO 3: Create vocabulary and Bag of Words
#
# # Fit vectorizer on training data
# vectorizer.fit(???)
#
# # Get the vocabulary
# list_of_words = vectorizer.get_feature_names_out()
# print(f"Vocabulary size: {len(list_of_words)}")
# print(f"Sample words: {list(list_of_words[:10])}")
#
# # Transform training data to Bag of Words
# train_data_bow = vectorizer.transform(???)
# print(f"\nBag of Words shape: {train_data_bow.shape}")
#
# # Compare original text vs BOW representation
# print(f"\nOriginal: '{train_data[1]}'")
# print(f"BOW vector: {train_data_bow[1].toarray()[0]}")
# print(f"Label: {train_label[1]}")
#
# YOUR CODE HERE:
vectorizer.fit(train_data)

# Get the vocabulary
list_of_words = vectorizer.get_feature_names_out()
print(f"Vocabulary size: {len(list_of_words)}")
print(f"Sample words: {list(list_of_words[:10])}")

train_data_bow = vectorizer.transform(train_data)
print(f"\nBag of Words shape: {train_data_bow.shape}")

print(f"\nOriginal: '{train_data[1]}'")
print(f"BOW vector: {train_data_bow[1].toarray()[0]}")
print(f"Label: {train_label[1]}")

Vocabulary size: 93
Sample words: ['about', 'afternoon', 'age', 'am', 'ani', 'apolog', 'appreci', 'are', 'around', 'assist']

Bag of Words shape: (80, 93)

Original: 'hi there'
BOW vector: [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
Label: Greeting


---

## Phase 5: Train ML Models 🤖

Train 3 classifiers to compare performance!

In [23]:
# ✅ WORKING EXAMPLE: Import classifiers
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import MultinomialNB

print("✅ All classifiers imported!")

✅ All classifiers imported!


### ⚠️ TODO 4: Train All 3 Classifiers (Easy)

**Your Task:** Create and train KNN, Decision Tree, and Naive Bayes classifiers.

**Steps:**
1. Create clf_knn with n_neighbors=5
2. Create clf_dt with random_state=33
3. Create clf_nb (MultinomialNB)
4. Fit each on train_data_bow and train_label

In [24]:
# ⚠️ TODO 4: Train all 3 classifiers
#
# # KNN Classifier
# clf_knn = KNeighborsClassifier(n_neighbors=???)
# clf_knn.fit(???, ???)
# print("✅ KNN trained!")
#
# # Decision Tree Classifier
# clf_dt = DecisionTreeClassifier(random_state=???)
# clf_dt.fit(???, ???)
# print("✅ Decision Tree trained!")
#
# # Naive Bayes Classifier
# clf_nb = MultinomialNB()
# clf_nb.fit(???, ???)
# print("✅ Naive Bayes trained!")
#
# YOUR CODE HERE:
clf_knn = KNeighborsClassifier(n_neighbors=3)
clf_knn.fit(train_data_bow, train_label)
print("✅ KNN trained!")

clf_dt = DecisionTreeClassifier(random_state=42)
clf_dt.fit(train_data_bow, train_label)
print("✅ Decision Tree trained!")

clf_nb = MultinomialNB()
clf_nb.fit(train_data_bow, train_label)
print("✅ Naive Bayes trained!")

✅ KNN trained!
✅ Decision Tree trained!
✅ Naive Bayes trained!


---

## Phase 6: Test Models 🧪

### ⚠️ TODO 5: Preprocess and Predict Test Sentence (Medium)

**Your Task:** Test all 3 models with "Hello there".

**Steps:**
1. Preprocess test_sentence
2. Transform to BOW (remember: put in a list!)
3. Predict with each classifier

**Expected:** All 3 should predict "Greeting"

In [25]:
# ⚠️ TODO 5: Test models with a sample sentence
#
# test_sentence = "Hello there"
#
# # Step 1: Preprocess
# test_processed = text_preprocessing(???)
# print(f"Preprocessed: '{test_processed}'")
#
# # Step 2: Transform to BOW (must be in a list!)
# test_bow = vectorizer.transform([???])
#
# # Step 3: Predict with each model
# print(f"\nKNN predicts: {clf_knn.predict(test_bow)[0]}")
# print(f"Decision Tree predicts: {clf_dt.predict(test_bow)[0]}")
# print(f"Naive Bayes predicts: {clf_nb.predict(test_bow)[0]}")
#
# YOUR CODE HERE:
pass
test_sentence = "Hello there"

test_processed = text_preprocessing(test_sentence)
print(f"Preprocessed: '{test_processed}'")

test_bow = vectorizer.transform([test_processed])

print(f"\nKNN predicts: {clf_knn.predict(test_bow)[0]}")
print(f"Decision Tree predicts: {clf_dt.predict(test_bow)[0]}")
print(f"Naive Bayes predicts: {clf_nb.predict(test_bow)[0]}")

Preprocessed: 'hello there'

KNN predicts: Greeting
Decision Tree predicts: Greeting
Naive Bayes predicts: Greeting


---

## Phase 7: Build Chatbot Interface 💬

### ⚠️ TODO 6: Complete bot_respond Function (Hard)

**Your Task:** Complete the bot_respond function that:
1. Preprocesses user input
2. Converts to BOW
3. Predicts intent
4. Returns appropriate response

**Hint:** Use clf_nb as the classifier (best for text)

In [26]:
# ⚠️ TODO 6: Complete the bot_respond function
#
# def bot_respond(user_query):
#     # Step 1: Preprocess user query
#     user_query = text_preprocessing(???)
#
#     # Step 2: Convert to Bag of Words
#     user_query_bow = vectorizer.transform([???])
#
#     # Step 3: Choose classifier (Naive Bayes works best for text)
#     clf = clf_nb
#
#     # Step 4: Predict intent
#     predicted = clf.predict(???)
#
#     # Handle low confidence predictions
#     max_proba = max(clf.predict_proba(user_query_bow)[0])
#     if max_proba < 0.08:
#         predicted = ['noanswer']
#
#     # Step 5: Select random response for predicted intent
#     numOfResponses = len(responses[predicted[0]])
#     chosenResponse = random.randint(0, numOfResponses-1)
#
#     # Handle time query specially (use datetime directly)
#     if predicted[0] == "TimeQuery":
#         bot_response = f"It's {datetime.now().strftime('%H:%M:%S')}"
#     else:
#         bot_response = responses[predicted[0]][chosenResponse]
#
#     return bot_response
#
# # Test the function
# print("Bot:", bot_respond("Hello there"))
# print("Bot:", bot_respond("What time is it?"))
# print("Bot:", bot_respond("Thank you!"))
#
# YOUR CODE HERE:
pass
def bot_respond(user_query):
     # Step 1: Preprocess user query
     user_query = text_preprocessing(user_query)

     # Step 2: Convert to Bag of Words
     user_query_bow = vectorizer.transform([user_query])

     # Step 3: Choose classifier (Naive Bayes works best for text)
     clf = clf_nb

     # Step 4: Predict intent
     predicted = clf.predict(user_query_bow)

     # Handle low confidence predictions
     max_proba = max(clf.predict_proba(user_query_bow)[0])
     if max_proba < 0.08:
         predicted = ['noanswer']

     # Step 5: Select random response for predicted intent
     numOfResponses = len(responses[predicted[0]])
     chosenResponse = random.randint(0, numOfResponses-1)

     # Handle time query specially (use datetime directly)
     if predicted[0] == "TimeQuery":
         bot_response = f"It's {datetime.now().strftime('%H:%M:%S')}"
     else:
        bot_response = responses[predicted[0]][chosenResponse]

     return bot_response

# Test the function
print("Bot:", bot_respond("Hello there"))
print("Bot:", bot_respond("What time is it?"))
print("Bot:", bot_respond("Thank you!"))

Bot: Hi human, please tell me your Alex user
Bot: It's 09:54:20
Bot: Anytime!


In [ ]:
# ✅ WORKING EXAMPLE: Simple chatbot interface
# Type your own message directly below and press Enter.
# Use 'exit', 'quit', or 'bye' to stop the chat.

print("This is Alex the chatbot. Say something!!")

while True:
    bot_input = input("You  : ")
    if bot_input.strip().lower() in ["exit", "quit", "bye"]:
        print("Bot  : Goodbye!")
        break
    print("Bot  :", bot_respond(bot_input))

This is Alex the chatbot. Say something!!
You  : Hello there
Bot  : Hello human, please tell me your Alex user
You  : What time is it?
Bot  : It's 09:54:29
You  : Thank you!
Bot  : Glad I could help!


## 🚀 Extension Challenges

In [28]:
# Challenge 1: Add username personalization
# Modify bot_respond to replace <HUMAN> with actual username

# Challenge 2: Compare model accuracy
# from sklearn.model_selection import train_test_split
# x_train, x_test, y_train, y_test = train_test_split(train_data_bow, train_label, test_size=0.2)
# Print accuracy for each model

print("Try the extension challenges after completing all TODOs!")

Try the extension challenges after completing all TODOs!


---

## 📤 Submission

**[Submit your completed project here](https://forms.gle/ynKFsTNGwHV4dp2s9)**